# 01 — Data Extraction & Profiling
**Purpose:** Load all raw CSVs, audit structure, flag quality issues, and produce a baseline extraction report.  
**Output:** `data/raw/*.csv` confirmed loaded · `reports/01_extraction_report.txt`  
**Next step:** `02_cleaning.ipynb`

## 1.0 — Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

RAW_DIR   = Path('data/raw')
CLEAN_DIR = Path('data/clean')
PROD_DIR  = Path('data/production')
REPORT_DIR = Path('reports')

for d in [RAW_DIR, CLEAN_DIR, PROD_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

## 1.1 — Define Source Files

In [ ]:
# Map of table name -> filename
SOURCE_FILES = {
    'circuits':              'circuits.csv',
    'constructorResults':    'constructorResults.csv',
    'constructorStandings':  'constructorStandings.csv',
    'constructors':          'constructors.csv',
    'driverStandings':       'driverStandings.csv',
    'drivers':               'drivers.csv',
    'lapTimes':              'lapTimes.csv',
    'pitStops':              'pitStops.csv',
    'qualifying':            'qualifying.csv',
    'races':                 'races.csv',
    'results':               'results.csv',
    'seasons':               'seasons.csv',
    'status':                'status.csv',
}

SOURCE_DIR = Path('data/raw')   # <-- point this to your source folder

# Sentinel values used in this dataset to represent nulls
NULL_SENTINELS = ['\\N', 'NULL', '', 'N/A', 'n/a']

## 1.2 — Load All Raw Files

In [ ]:
raw = {}
load_log = []

for name, fname in SOURCE_FILES.items():
    fpath = SOURCE_DIR / fname
    try:
        df = pd.read_csv(
            fpath,
            encoding='latin1',
            na_values=NULL_SENTINELS,
            keep_default_na=True
        )
        raw[name] = df
        load_log.append({'table': name, 'rows': len(df), 'cols': len(df.columns), 'status': 'OK'})
        print(f'  [OK] {name:30s} {len(df):>7,} rows  {len(df.columns)} cols')
    except Exception as e:
        load_log.append({'table': name, 'rows': 0, 'cols': 0, 'status': str(e)})
        print(f'  [FAIL] {name}: {e}')

load_summary = pd.DataFrame(load_log)
print(f'\nLoaded {load_summary[load_summary.status=="OK"].shape[0]}/{len(SOURCE_FILES)} tables successfully.')

## 1.3 — Schema Audit (columns, dtypes, nulls)

In [ ]:
def audit_table(name, df):
    """Return per-column audit row for a dataframe."""
    rows = []
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct   = round(null_count / len(df) * 100, 1)
        n_unique   = df[col].nunique(dropna=True)
        sample     = df[col].dropna().iloc[0] if null_count < len(df) else 'ALL NULL'
        rows.append({
            'table':       name,
            'column':      col,
            'dtype':       str(df[col].dtype),
            'null_count':  null_count,
            'null_pct':    null_pct,
            'n_unique':    n_unique,
            'sample_val':  str(sample)[:60]
        })
    return rows

audit_rows = []
for name, df in raw.items():
    audit_rows.extend(audit_table(name, df))

audit_df = pd.DataFrame(audit_rows)

# Highlight high-null columns (>50%)
high_null = audit_df[audit_df['null_pct'] > 50]
print(f'Columns with >50% nulls: {len(high_null)}')
display(high_null[['table','column','null_count','null_pct']])

## 1.4 — Duplicate Row Check

In [ ]:
dup_report = []
for name, df in raw.items():
    n_dups = df.duplicated().sum()
    dup_report.append({'table': name, 'duplicate_rows': n_dups})
    if n_dups > 0:
        print(f'  [WARN] {name}: {n_dups} duplicate rows found')

dup_df = pd.DataFrame(dup_report)
print('\nDuplicate summary:')
display(dup_df)

## 1.5 — Primary Key Integrity Check

In [ ]:
# Define expected primary keys per table
PK_MAP = {
    'circuits':             ['circuitId'],
    'constructorResults':   ['constructorResultsId'],
    'constructorStandings': ['constructorStandingsId'],
    'constructors':         ['constructorId'],
    'driverStandings':      ['driverStandingsId'],
    'drivers':              ['driverId'],
    'lapTimes':             ['raceId', 'driverId', 'lap'],
    'pitStops':             ['raceId', 'driverId', 'stop'],
    'qualifying':           ['qualifyId'],
    'races':                ['raceId'],
    'results':              ['resultId'],
    'seasons':              ['year'],
    'status':               ['statusId'],
}

pk_report = []
for name, pk_cols in PK_MAP.items():
    if name not in raw:
        continue
    df = raw[name]
    missing_cols = [c for c in pk_cols if c not in df.columns]
    if missing_cols:
        pk_report.append({'table': name, 'pk': str(pk_cols), 'duplicates': 'N/A — col missing', 'nulls': 'N/A'})
        continue
    n_dups  = df.duplicated(subset=pk_cols).sum()
    n_nulls = df[pk_cols].isnull().any(axis=1).sum()
    pk_report.append({'table': name, 'pk': str(pk_cols), 'duplicates': n_dups, 'nulls': n_nulls})
    if n_dups > 0 or n_nulls > 0:
        print(f'  [WARN] {name} PK issue — dups: {n_dups}, nulls: {n_nulls}')

display(pd.DataFrame(pk_report))

## 1.6 — Referential Integrity Check (FK → PK)

In [ ]:
# Check that foreign keys in child tables resolve to parent tables
FK_CHECKS = [
    ('results',              'raceId',        'races',        'raceId'),
    ('results',              'driverId',       'drivers',      'driverId'),
    ('results',              'constructorId',  'constructors', 'constructorId'),
    ('results',              'statusId',       'status',       'statusId'),
    ('races',                'circuitId',      'circuits',     'circuitId'),
    ('lapTimes',             'raceId',         'races',        'raceId'),
    ('pitStops',             'raceId',         'races',        'raceId'),
    ('qualifying',           'raceId',         'races',        'raceId'),
    ('driverStandings',      'raceId',         'races',        'raceId'),
    ('constructorStandings', 'raceId',         'races',        'raceId'),
]

fk_report = []
for child_tbl, fk_col, parent_tbl, pk_col in FK_CHECKS:
    if child_tbl not in raw or parent_tbl not in raw:
        continue
    child_vals  = set(raw[child_tbl][fk_col].dropna().unique())
    parent_vals = set(raw[parent_tbl][pk_col].dropna().unique())
    orphans     = child_vals - parent_vals
    fk_report.append({
        'child':   f'{child_tbl}.{fk_col}',
        'parent':  f'{parent_tbl}.{pk_col}',
        'orphan_values': len(orphans),
        'sample_orphans': str(list(orphans)[:5]) if orphans else 'None'
    })

display(pd.DataFrame(fk_report))

## 1.7 — Value Range & Sanity Checks

In [ ]:
# Races: year range expected 1950-2018
races = raw['races']
print('Race year range:', races['year'].min(), '–', races['year'].max())
print('Rounds per year:\n', races.groupby('year')['round'].max().tail(10))

# Results: points should be non-negative
results = raw['results']
print('\nNegative points:', (results['points'] < 0).sum())
print('Points value counts (top 10):', results['points'].value_counts().head(10).to_dict())

# Pit stops: duration outliers
pit = raw['pitStops']
pit['ms_numeric'] = pd.to_numeric(pit['milliseconds'], errors='coerce')
print('\nPit stop ms — min/max/mean:', pit['ms_numeric'].min(), '/', pit['ms_numeric'].max(), '/', round(pit['ms_numeric'].mean()))
print('Suspiciously long stops (>120s):',  (pit['ms_numeric'] > 120_000).sum())
print('Suspiciously short stops (<5s):',   (pit['ms_numeric'] < 5_000).sum())

# Lap times: outlier check
lt = raw['lapTimes']
print('\nLap time ms — min/max/mean:', lt['milliseconds'].min(), '/', lt['milliseconds'].max(), '/', round(lt['milliseconds'].mean()))

## 1.8 — Coverage Summary (year-by-table)

In [ ]:
# Which tables have data for which year ranges? Important for era-based analysis.
races_ref = raw['races'][['raceId','year']]

tables_with_raceId = ['results', 'lapTimes', 'pitStops', 'qualifying',
                       'driverStandings', 'constructorStandings', 'constructorResults']

for tbl in tables_with_raceId:
    if tbl not in raw:
        continue
    merged = raw[tbl][['raceId']].merge(races_ref, on='raceId', how='left')
    year_min = merged['year'].min()
    year_max = merged['year'].max()
    print(f'  {tbl:30s}  years: {year_min} – {year_max}  rows: {len(raw[tbl]):>8,}')

## 1.9 — Export Extraction Report

In [ ]:
report_path = REPORT_DIR / '01_extraction_report.csv'
audit_df.to_csv(report_path, index=False)
print(f'Audit report saved → {report_path}')
print(f'Total columns audited: {len(audit_df)}')
print(f'High-null columns (>50%): {len(high_null)}')
display(load_summary)